# Machine Learning Network Intrusion Detection

This notebook contains the reproducible workflow used to compare Random Forest and LightGBM classifiers on the CIC-IDS2017 network intrusion dataset.

The dataset is publicly available from the Canadian Institute for Cybersecurity and is intentionally not included in this repository.


## Workflow

- Load selected CIC-IDS2017 CSV files
- Clean missing/infinite values and duplicates
- Convert labels to Normal/Malicious
- Stratified 60/20/20 split
- Correlation and variance-based feature selection
- PCA exploratory analysis
- Train Random Forest and LightGBM baselines
- Tune both models with RandomizedSearchCV
- Retrain the selected model on train + validation data
- Evaluate on the held-out test set


## Dataset

Download CIC-IDS2017 separately from: https://www.unb.ca/cic/datasets/ids-2017.html

The original experiment used selected Tuesday-Friday CSV files and excluded Monday's benign-only traffic and the Friday afternoon PortScan CSV.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score
from lightgbm import LGBMClassifier


In [ ]:
# Example: load the locally downloaded CIC-IDS2017 CSV files
from pathlib import Path
DATA_DIR = Path('CSVs')
csv_files = list(DATA_DIR.glob('*.csv'))
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df.columns = df.columns.str.strip()
df['Label'] = df['Label'].apply(lambda x: 0 if x == 'BENIGN' else 1)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)


In [ ]:
features = df.drop('Label', axis=1)
labels = df['Label']
X_train, X_temp, y_train, y_temp = train_test_split(features, labels, test_size=0.4, stratify=labels, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

corr_matrix = X_train.corr()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
correlated_features = [c for c in upper.columns if any(upper[c] > 0.95)]
X_train = X_train.drop(columns=correlated_features)
X_val = X_val.drop(columns=correlated_features)
X_test = X_test.drop(columns=correlated_features)

selector = VarianceThreshold(threshold=0.01)
X_train = selector.fit_transform(X_train)
X_val = selector.transform(X_val)
X_test = selector.transform(X_test)


In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42)
rf_model.fit(X_train, y_train)

lgbm_model = LGBMClassifier(n_estimators=500, learning_rate=0.05, is_unbalance=True, random_state=42, n_jobs=-1)
lgbm_model.fit(X_train, y_train)

for name, model in [('Random Forest', rf_model), ('LightGBM', lgbm_model)]:
    pred = model.predict(X_val)
    print(name)
    print(classification_report(y_val, pred, digits=4))


## Final model

The original evaluation selected the LightGBM baseline as the final model. It was retrained on the combined training and validation sets before evaluation on the untouched test set. The original run achieved an F1-score of approximately 99.9%, with 15 false negatives and 107 false positives.
